In [ ]:
!uv add pandas
!uv add scikit-learn

Resolved 19 packages in 14ms
Checked 17 packages in 3ms
Resolved 19 packages in 4ms
Checked 17 packages in 1ms
Resolved 19 packages in 5ms
Checked 17 packages in 1ms


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [48]:
df = pd.read_csv("./data/obesity_data.csv")
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [49]:
# add BMI col
df["BMI"] = df["Weight"] / (df["Height"] ** 2)
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,BMI
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight,24.386526
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight,24.238227
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight,23.765432
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I,26.851852
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II,28.342381


In [50]:
# split features and target
x = df.drop("NObeyesdad", axis=1)
y = df["NObeyesdad"]

In [51]:
# encode target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# print classes
for idx, cls in enumerate(label_encoder.classes_):
  print(f"{cls}: {idx}")

Insufficient_Weight: 0
Normal_Weight: 1
Obesity_Type_I: 2
Obesity_Type_II: 3
Obesity_Type_III: 4
Overweight_Level_I: 5
Overweight_Level_II: 6


In [52]:
# identify numerical and categorical cols
num_cols = x.select_dtypes(include=["int64", "float64"]).columns
cat_cols = x.select_dtypes(include=["object", "str"]).columns

print(f"num_cols: {num_cols}")
print(f"cat_cols: {cat_cols}")

num_cols: Index(['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'BMI'], dtype='str')
cat_cols: Index(['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE',
       'SCC', 'CALC', 'MTRANS'],
      dtype='str')


In [53]:
# standardize num_cols with standard_scaler
standard_scaler = StandardScaler()
x_num_cols_df = pd.DataFrame(
    standard_scaler.fit_transform(x[num_cols]),
    columns=num_cols
)

# one hot encode cat_cols with onehotencoder
onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
x_cat_cols_df = pd.DataFrame(
    onehot_encoder.fit_transform(x[cat_cols]),
    columns=onehot_encoder.get_feature_names_out(cat_cols)
)

In [54]:
# combine num_cols and cat_cols
x_preprocessed = pd.concat(
    [x_num_cols_df, x_cat_cols_df],
    axis=1
)

In [56]:
# split the data and write to a csv file
x_train, x_test, y_train, y_test = train_test_split(
    x_preprocessed,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

import os
os.makedirs("./data/preprocessed", exist_ok=True)

x_train.to_csv("./data/preprocessed/x_train.csv", index=False)
x_test.to_csv("./data/preprocessed/x_test.csv", index=False)

y_train_df = pd.DataFrame(y_train, columns=["NObeyesdad"])
y_train_df.to_csv("./data/preprocessed/y_train.csv", index=False)

y_test_df = pd.DataFrame(y_test, columns=["NObeyesdad"])
y_test_df.to_csv("./data/preprocessed/y_test.csv", index=False)


In [57]:
# label mappings for reference
label_mapping = pd.DataFrame({
    "class_name": label_encoder.classes_,
    "encoded_value": range(len(label_encoder.classes_))
})

label_mapping.to_csv(
    "./data/preprocessed/label_mapping.csv",
    index=False
)